# Independent accuracy checks and simple benchmark

Run notebook 12 first to generate reports. This notebook has its own kernel
setup and checks exported computations against the source. Passing the audit
establishes consistency, not guaranteed future accuracy. June was reused.

## How this extends the team submission

Notebooks 02–03 predict tomorrow; this contribution forecasts each day of the
next week together and refits weekly. It does not replace root-cause, offer,
product-stock or Garden profit-outlook analyses. Notebook 13 reuses notebook
04's hypothetical shortage/excess penalties on matching week-ahead forecasts.
The existing next-day XGBoost scores use a different protocol and are not a
controlled head-to-head comparison. June remains exploratory.


In [1]:
from pathlib import Path
import sys
import os
import json
import hashlib
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Set this to a path if the team stores its CSV somewhere else.
DATA_OVERRIDE = os.environ.get("STORIES_DATA")
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "src/stories_coffee/pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Copy src/stories_coffee/ alongside notebooks/ first.")
sys.path.insert(0, str(ROOT / "src"))
import stories_coffee
if Path(stories_coffee.__file__).resolve().parent != ROOT / "src/stories_coffee":
    raise RuntimeError("Another stories_coffee package is loaded. Restart the kernel.")
if DATA_OVERRIDE is not None:
    DATA = Path(DATA_OVERRIDE).expanduser()
    if not DATA.is_absolute():
        DATA = ROOT / DATA
else:
    candidates = [ROOT / p for p in (
        "data/raw/stories_transactions.csv", "data/stories_transactions.csv",
        "stories_transactions.csv") if (ROOT / p).is_file()]
    if len(candidates) != 1:
        raise FileNotFoundError("Set DATA_OVERRIDE: expected exactly one source CSV.")
    DATA = candidates[0]
if not DATA.is_file():
    raise FileNotFoundError(DATA)
OUT = ROOT / "artifacts/week_ahead_forecasting"
PUBLIC = ROOT / "reports/week_ahead_forecasting"
OUT.mkdir(parents=True, exist_ok=True)
print("Source:", DATA.relative_to(ROOT) if DATA.is_relative_to(ROOT) else DATA.name)
print("SHA256:", hashlib.sha256(DATA.read_bytes()).hexdigest())
print("Outputs: artifacts/week_ahead_forecasting/")

Source: stories_transactions.csv
SHA256: f1672e728c28295b2b802d878fd38a158ba039827b429355e8438ee0463237a1
Outputs: artifacts/week_ahead_forecasting/


In [2]:
if not (OUT / "summary.json").is_file():
    raise FileNotFoundError("Run notebook 12 first to create forecasting results.")
from stories_coffee.validation import validate
audit = validate(DATA, OUT)
display(pd.DataFrame([{k: audit[k] for k in ["status", "checks_run", "scope"]}]))
assert audit["status"] == "PASS", audit["failures"]

,status,checks_run,scope
0,PASS,20223,"Source targets, selection, coefficients, blend..."


## Recompute June errors from exported predictions

The independent audit also checks source targets, training cutoffs, selection,
saved coefficients and error arithmetic. Recompute aggregate metrics here;
do not interpret the passing checks as new independent observations.

In [3]:
predictions = pd.read_csv(OUT / "tables/predictions.csv", parse_dates=["date", "origin"])
june = predictions[predictions.stage.eq("june")].copy()
assert (june.date >= june.origin).all()
assert not june.duplicated(["method", "date", "branch"]).any()
def metrics(frame):
    errors = frame.predicted - frame.actual
    wape = errors.abs().sum() / frame.actual.sum()
    return {"n": len(frame), "WAPE (%)": 100 * wape,
            "100-WAPE (%)": 100 * (1 - wape), "MAE": errors.abs().mean(),
            "RMSE": np.sqrt((errors ** 2).mean()),
            "Bias (%)": 100 * errors.sum() / frame.actual.sum()}
result = json.loads((OUT / "summary.json").read_text())
for method in ["baseline", "selected"]:
    measured = metrics(june[june.method.eq(method)])
    assert np.isclose(measured["WAPE (%)"], 100 * result[method]["wape"])

## Add a four-week seasonal-naive benchmark

For each target date, average the same weekday over the four weeks preceding
its forecast issue date. All four observations must be before that issue.
This forecasts a complete week without using new sales from within that week.
Evaluate on exactly the selected model's branch-days. This comparator is
reported diagnostically; it does not retune the model using June.

In [4]:
from stories_coffee.data import load_data
history = load_data(DATA).daily.set_index(["date", "branch"]).transactions
naive = june[june.method.eq("selected")].copy()
values = []
for row in naive.itertuples():
    past = [row.date - pd.Timedelta(weeks=k) for k in range(1, 5)]
    assert all(day < row.origin for day in past)
    values.append(np.mean([history.loc[(day, row.branch)] for day in past]))
naive["predicted"] = values
naive["method"] = "seasonal_naive_4_weeks"
comparison_rows = pd.concat([june, naive], ignore_index=True)
table = pd.DataFrame([{"method": name, **metrics(frame)}
                      for name, frame in comparison_rows.groupby("method")])
display(table.round(4))
table.to_csv(OUT / "tables/notebook_benchmark_metrics.csv", index=False)
naive.to_csv(OUT / "tables/notebook_naive_predictions.csv", index=False)

,method,n,WAPE (%),100-WAPE (%),MAE,RMSE,Bias (%)
0,baseline,90,9.0444,90.9556,6.9401,8.8587,0.4152
1,seasonal_naive_4_weeks,90,9.8936,90.1064,7.5917,9.9889,-4.4345
2,selected,90,8.8277,91.1723,6.7738,8.9224,-1.2520


## Inspect where errors concentrate

The overall WAPE weights high-volume branches more heavily. Show branch
and weekday errors so a strong company score cannot hide a weak branch.
Thirty June days leave few observations per weekday, so these breakdowns
are descriptive rather than evidence for new feature tuning.

In [5]:
selected = june[june.method.eq("selected")].copy()
branch_table = pd.DataFrame([{"branch": branch, **metrics(frame)}
                            for branch, frame in selected.groupby("branch")])
selected["weekday"] = selected.date.dt.day_name()
weekday_table = pd.DataFrame([{"weekday": day, **metrics(frame)}
                             for day, frame in selected.groupby("weekday")])
display(branch_table.round(3))
display(weekday_table.round(3))
branch_table.to_csv(OUT / "tables/notebook_branch_metrics.csv", index=False)
weekday_table.to_csv(OUT / "tables/notebook_weekday_metrics.csv", index=False)

,branch,n,WAPE (%),100-WAPE (%),MAE,RMSE,Bias (%)
0,Garden,30,8.965,91.035,6.727,8.463,-0.588
1,Harbour,30,7.189,92.811,8.114,10.582,-2.230
2,Uptown,30,12.957,87.043,5.481,7.432,0.180


,weekday,n,WAPE (%),100-WAPE (%),MAE,RMSE,Bias (%)
0,Friday,12,11.157,88.843,9.242,11.599,-4.785
1,Monday,15,7.128,92.872,5.208,6.724,-3.755
2,Saturday,12,10.716,89.284,5.421,6.924,2.344
3,Sunday,12,7.386,92.614,4.696,6.857,-2.194
4,Thursday,12,6.212,93.788,4.462,5.739,0.614
5,Tuesday,15,7.875,92.125,9.025,11.470,-1.810
6,Wednesday,12,12.749,87.251,9.190,10.662,3.538


## What would establish better results?

The selected blend improves June absolute error slightly over the regression
baseline, but does not improve every metric. Five issue blocks and repeated
inspection limit inference. Keep the selected recipe fixed; save forecasts
before new sales arrive and score the same branch-days for every benchmark.

For business validation, compare forecast-assisted and normal planning under
a pre-specified pilot, measuring paid labour, service delays and availability.
A 91% WAPE-based score is neither 91% confidence nor proof of profit gains.
This contribution evaluates point forecasts, not calibrated prediction ranges.

## Connect forecasts to the team's existing decision-cost framework

Reuse `forecast_business_value.transaction_economics` and `decision_loss`
without calling its report generator or changing its models. All nine original
shortage/excess penalty combinations are evaluated, with integer capacity and
continuous sensitivity. Jan–May economics price June errors. No June-based
policy selection is performed. Compare the selected forecast, regression
baseline and four-week naive under the same weekly issue schedule.

These are hypothetical transaction-capacity penalties, not measured savings,
SKU orders or actual payroll. Scenario wins reuse the same errors and are not
independent experiments. Missing next-day prediction artifacts are unnecessary.


In [6]:
from week_ahead_integration import decision_study, publish
decision = decision_study(DATA, comparison_rows, ROOT, OUT)
display(decision[decision.Branch.eq("ALL")].round(2))
publish(ROOT, OUT, PUBLIC, table, branch_table, weekday_table, decision, audit)
print("Aggregate evidence: reports/week_ahead_forecasting/")
print("Detailed forecasts and coefficient files remain in ignored artifacts/.")


,Scenario,ShortageFraction,WasteFraction,CapacityPolicy,method,Branch,ShortageTransactions,ExcessTransactions,ShortageCostUSD,OverPreparationCostUSD,TotalDecisionCostUSD
3,S0.25_W0.1,0.25,0.10,integer,baseline,ALL,299.00,326.00,397.10,122.58,519.68
7,S0.25_W0.1,0.25,0.10,continuous,baseline,ALL,297.97,326.64,395.79,122.94,518.73
11,S0.25_W0.1,0.25,0.10,integer,seasonal_naive_4_weeks,ALL,489.00,193.00,653.21,73.97,727.19
15,S0.25_W0.1,0.25,0.10,continuous,seasonal_naive_4_weeks,ALL,494.75,188.50,660.73,72.40,733.13
19,S0.25_W0.1,0.25,0.10,integer,selected,ALL,348.00,263.00,461.35,99.35,560.70
23,S0.25_W0.1,0.25,0.10,continuous,selected,ALL,348.05,261.59,461.50,98.95,560.45
27,S0.25_W0.25,0.25,0.25,integer,baseline,ALL,299.00,326.00,397.10,306.45,703.55
31,S0.25_W0.25,0.25,0.25,continuous,baseline,ALL,297.97,326.64,395.79,307.36,703.15
35,S0.25_W0.25,0.25,0.25,integer,seasonal_naive_4_weeks,ALL,489.00,193.00,653.21,184.93,838.15
39,S0.25_W0.25,0.25,0.25,continuous,seasonal_naive_4_weeks,ALL,494.75,188.50,660.73,181.01,841.74


Aggregate evidence: reports/week_ahead_forecasting/
Detailed forecasts and coefficient files remain in ignored artifacts/.


## Main decision tradeoff from this run

The selected blend has lower June WAPE, but the regression baseline has lower
assumed preparation cost in eight of the nine scenarios. Against the four-week
naive forecast, the blend has lower assumed cost in all nine. Integer and
continuous capacity give the same scenario win counts. This illustrates the
team's central point: lower prediction error does not guarantee higher profit.
These comparisons do not establish actual savings or justify a new policy
selected after seeing June. Retain both regression recipes in the prospective
pilot and verify the operational penalties before choosing one.
